# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [2]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [3]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [4]:
#your code here
spaceship.dropna(inplace=True)
spaceship.drop_duplicates(inplace=True)
spaceship['Cabin'] = spaceship['Cabin'].str[0]
spaceship.drop(columns=['PassengerId','Name'],inplace=True)
spaceship = pd.get_dummies(
    spaceship,
    columns=['HomePlanet','CryoSleep','Cabin','Destination','VIP'],
    drop_first=True,
    dtype=int
)
spaceship.head()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
0,39.0,0.0,0.0,0.0,0.0,0.0,False,1,0,0,1,0,0,0,0,0,0,0,1,0
1,24.0,109.0,9.0,25.0,549.0,44.0,True,0,0,0,0,0,0,0,1,0,0,0,1,0
2,58.0,43.0,3576.0,0.0,6715.0,49.0,False,1,0,0,0,0,0,0,0,0,0,0,1,1
3,33.0,0.0,1283.0,371.0,3329.0,193.0,False,1,0,0,0,0,0,0,0,0,0,0,1,0
4,16.0,303.0,70.0,151.0,565.0,2.0,True,0,0,0,0,0,0,0,1,0,0,0,1,0


In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

spaceship[['Age','RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']] = scaler.fit_transform(spaceship[['Age','RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']])

spaceship.head()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
0,0.695413,-0.345756,-0.285355,-0.309494,-0.273759,-0.269534,False,1,0,0,1,0,0,0,0,0,0,0,1,0
1,-0.336769,-0.176748,-0.279993,-0.266112,0.206165,-0.230494,True,0,0,0,0,0,0,0,1,0,0,0,1,0
2,2.002842,-0.279083,1.845163,-0.309494,5.596357,-0.226058,False,1,0,0,0,0,0,0,0,0,0,0,1,1
3,0.282540,-0.345756,0.479034,0.334285,2.636384,-0.098291,False,1,0,0,0,0,0,0,0,0,0,0,1,0
4,-0.887266,0.124056,-0.243650,-0.047470,0.220152,-0.267759,True,0,0,0,0,0,0,0,1,0,0,0,1,0


- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [6]:
#your code here
X = spaceship.drop('Transported',axis=1)
y = spaceship['Transported']
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
print(X_train.shape)
print(X_test.shape)

(5284, 19)
(1322, 19)


In [7]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

gb.fit(X_train, y_train)

,loss,'log_loss'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


- Evaluate your model

In [8]:
#your code here
from sklearn.metrics import classification_report
from sklearn.metrics import roc_auc_score
from sklearn.metrics import accuracy_score

gb_y_pred = gb.predict(X_test)
gb_y_prob = gb.predict_proba(X_test)[:,1]

print("Gradient Boosting")
print("Accuracy:", accuracy_score(y_test,gb_y_pred))
print("ROC-AUC:", roc_auc_score(y_test,gb_y_prob))
print(classification_report(y_test,gb_y_pred))

Gradient Boosting
Accuracy: 0.8071104387291982
ROC-AUC: 0.8896618344217901
              precision    recall  f1-score   support

       False       0.84      0.76      0.79       653
        True       0.78      0.86      0.82       669

    accuracy                           0.81      1322
   macro avg       0.81      0.81      0.81      1322
weighted avg       0.81      0.81      0.81      1322



**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [9]:
#your code here
from sklearn.model_selection import GridSearchCV
param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

gb = GradientBoostingClassifier(
    random_state=42
)

grid_search = GridSearchCV(
    estimator=gb,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=5,
    n_jobs=-1,
    verbose=1
)


- Run Grid Search

In [10]:
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 243 candidates, totalling 1215 fits


,estimator,GradientBoost...ndom_state=42)
,param_grid,"{'learning_rate': [0.01, 0.05, ...], 'max_depth': [3, 5, ...], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], ...}"
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,loss,'log_loss'


In [11]:
print("Best Parameters:")
print(grid_search.best_params_)

print("Best CV ROC-AUC:")
print(grid_search.best_score_)

Best Parameters:
{'learning_rate': 0.05, 'max_depth': 3, 'min_samples_leaf': 4, 'min_samples_split': 2, 'n_estimators': 200}
Best CV ROC-AUC:
0.8843478330051184


- Evaluate your model

In [12]:
best_gb = grid_search.best_estimator_
y_pred = best_gb.predict(X_test)
y_prob = best_gb.predict_proba(X_test)[:,1]

print("Tuned Gradient Boosting")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))

Tuned Gradient Boosting
Accuracy: 0.8071104387291982
ROC-AUC: 0.8904881917881595
              precision    recall  f1-score   support

       False       0.84      0.76      0.79       653
        True       0.78      0.86      0.82       669

    accuracy                           0.81      1322
   macro avg       0.81      0.81      0.81      1322
weighted avg       0.81      0.81      0.81      1322



Hyperparameter tuning was conducted using GridSearchCV with 5-fold cross-validation. The tuned Gradient Boosting model achieved a ROC-AUC of 0.890, which was slightly higher than the baseline model (0.890 vs. 0.889). The results suggest that the default configuration was already close to optimal for this dataset.